# EdgeElevate - Complete End-to-End Workflow

This notebook implements the full EdgeElevate workflow using real API integrations.

## Workflow Steps:
1. **User Input** - Startup name and optional parameters
2. **Competitor Analysis** (Peec AI) - Identify top competitors and positioning
3. **Public Sentiment Analysis** (Tavily) - Scrape reviews from Trustpilot, G2
4. **Insight Structuring** (Q-Context) - Organize data into actionable insights
5. **Content Generation** (Hera + OpenRouter) - Video scripts and LinkedIn posts
6. **Report Generation** - Structured output with all deliverables

## Setup and Configuration

In [ ]:
import os
import json
import time
import requests
from dotenv import load_dotenv
from typing import Dict, List, Any, Optional
from dataclasses import dataclass, asdict

# Load environment variables
load_dotenv()

# API Configuration
@dataclass
class APIConfig:
    peec_key: str = os.getenv("PEEC_API_KEY", "")
    qcontext_key: str = os.getenv("QCONTEXT_API_KEY", "")
    hera_key: str = os.getenv("HERA_API_KEY", "")
    tavily_key: str = os.getenv("TAVILY_API_KEY", "")
    openrouter_key: str = os.getenv("OPENROUTER_API_KEY", "")
    
    peec_base: str = "https://api.peec.ai/customer/v1"
    qcontext_base: str = "https://api.qontext.ai/v1"
    hera_base: str = "https://api.hera.video"
    tavily_base: str = "https://api.tavily.com"
    openrouter_base: str = "https://openrouter.ai/api/v1"

config = APIConfig()

# Helper functions
def pretty_print(title: str, data: Any):
    """Pretty print JSON data"""
    print(f"\n{'='*60}")
    print(f" {title}")
    print(f"{'='*60}")
    if isinstance(data, (dict, list)):
        print(json.dumps(data, indent=2))
    else:
        print(data)

print("✅ Environment and configuration loaded!")

## Node 1: User Input

In [ ]:
@dataclass
class UserInput:
    startup_name: str
    industry: Optional[str] = None
    target_audience: Optional[str] = None
    region: Optional[str] = None

# User input
user_input = UserInput(
    startup_name="Nothing Phone",
    industry="Consumer Electronics",
    target_audience="Tech enthusiasts and design-conscious consumers",
    region="Global"
)

# Initialize workflow state
workflow_state = {
    "input": asdict(user_input),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
}

pretty_print("User Input", workflow_state["input"])

## Node 2: Competitor Analysis (Peec AI)

Identify top competitors and extract positioning insights

In [ ]:
def competitor_analysis_node(startup_name: str) -> Dict[str, Any]:
    """
    Use Peec AI to analyze competitors.
    Falls back to OpenRouter if Peec is not available.
    """
    
    # Option 1: Try Peec AI (if Enterprise access available)
    if config.peec_key:
        try:
            headers = {
                "Authorization": f"Bearer {config.peec_key}",
                "Content-Type": "application/json"
            }
            
            # Get brand report
            response = requests.get(
                f"{config.peec_base}/brands",
                headers=headers
            )
            
            if response.status_code == 200:
                return {
                    "source": "peec_ai",
                    "data": response.json()
                }
        except Exception as e:
            print(f"⚠️ Peec AI failed: {str(e)}. Falling back to OpenRouter...")
    
    # Option 2: Fallback to OpenRouter for competitor analysis
    if config.openrouter_key:
        try:
            headers = {
                "Authorization": f"Bearer {config.openrouter_key}",
                "Content-Type": "application/json"
            }
            
            prompt = f"""
            Analyze the competitive landscape for: {startup_name}
            
            Provide a detailed JSON response with:
            1. Top 3-5 primary competitors
            2. For each competitor:
               - Name
               - Market positioning
               - Key strengths
               - Key weaknesses
               - Differentiation factors
            3. Narrative gaps (opportunities for {startup_name})
            4. Market overview
            
            Return ONLY valid JSON, no markdown formatting.
            """
            
            payload = {
                "model": "google/gemini-2.0-flash-exp:free",
                "messages": [{"role": "user", "content": prompt}],
                "response_format": {"type": "json_object"}
            }
            
            response = requests.post(
                f"{config.openrouter_base}/chat/completions",
                headers=headers,
                json=payload
            )
            
            if response.status_code == 200:
                result = response.json()
                content = result["choices"][0]["message"]["content"]
                return {
                    "source": "openrouter_llm",
                    "data": json.loads(content)
                }
        except Exception as e:
            print(f"❌ OpenRouter failed: {str(e)}")
    
    return {"source": "none", "error": "No API available for competitor analysis"}

# Execute competitor analysis
competitor_data = competitor_analysis_node(user_input.startup_name)
workflow_state["competitor_analysis"] = competitor_data

pretty_print("Competitor Analysis Results", competitor_data)

## Node 3: Public Sentiment Analysis (Tavily)

Scrape and analyze reviews from Trustpilot, G2, and other sources

In [ ]:
def sentiment_analysis_node(startup_name: str, competitors: List[str]) -> Dict[str, Any]:
    """
    Use Tavily to search for reviews and sentiment data
    """
    
    if not config.tavily_key:
        return {"error": "Tavily API key not configured"}
    
    try:
        all_reviews = {}
        
        # Search for startup reviews
        for source in ["trustpilot", "g2", "reddit"]:
            payload = {
                "api_key": config.tavily_key,
                "query": f"{startup_name} {source} reviews customer feedback opinions",
                "search_depth": "advanced",
                "max_results": 5
            }
            
            response = requests.post(
                f"{config.tavily_base}/search",
                json=payload
            )
            
            if response.status_code == 200:
                results = response.json()
                all_reviews[f"{startup_name}_{source}"] = results.get("results", [])
        
        # Search for competitor reviews (top 2 competitors)
        for competitor in competitors[:2]:
            payload = {
                "api_key": config.tavily_key,
                "query": f"{competitor} reviews customer feedback",
                "search_depth": "basic",
                "max_results": 3
            }
            
            response = requests.post(
                f"{config.tavily_base}/search",
                json=payload
            )
            
            if response.status_code == 200:
                results = response.json()
                all_reviews[f"{competitor}_reviews"] = results.get("results", [])
        
        return {
            "source": "tavily",
            "reviews": all_reviews
        }
        
    except Exception as e:
        return {"error": f"Tavily search failed: {str(e)}"}

# Extract competitor names from previous node
competitors = []
if competitor_data.get("data") and "competitors" in competitor_data["data"]:
    competitors = [c.get("name", c) if isinstance(c, dict) else c 
                   for c in competitor_data["data"]["competitors"]]

# Execute sentiment analysis
sentiment_data = sentiment_analysis_node(user_input.startup_name, competitors)
workflow_state["sentiment_analysis"] = sentiment_data

pretty_print("Sentiment Analysis Results", sentiment_data)

## Node 4: Insight Structuring (Q-Context)

Organize raw data into structured insights and positioning narrative

In [ ]:
def insight_structuring_node(competitor_data: Dict, sentiment_data: Dict, startup_name: str) -> Dict[str, Any]:
    """
    Use Q-Context to structure insights from raw data
    Falls back to OpenRouter if Q-Context is not available
    """
    
    # Prepare data for ingestion
    raw_data = f"""
    STARTUP: {startup_name}
    
    COMPETITOR ANALYSIS:
    {json.dumps(competitor_data, indent=2)}
    
    SENTIMENT ANALYSIS:
    {json.dumps(sentiment_data, indent=2)}
    """
    
    # Option 1: Try Q-Context
    if config.qcontext_key:
        try:
            headers = {
                "X-API-Key": config.qcontext_key,
                "Content-Type": "application/json"
            }
            
            # Ingest data
            ingest_response = requests.post(
                f"{config.qcontext_base}/ingestion/unstructured",
                headers=headers,
                json={"text": raw_data}
            )
            
            if ingest_response.status_code == 201:
                # Retrieve structured insights
                retrieval_payload = {
                    "prompt": f"""
                    Based on the ingested data, provide structured insights for {startup_name}:
                    1. Core strengths
                    2. Key weaknesses
                    3. Competitor advantages
                    4. Market positioning opportunities
                    5. Refined value proposition
                    6. Differentiation angle
                    7. Target audience alignment
                    """,
                    "includeSources": True
                }
                
                retrieval_response = requests.post(
                    f"{config.qcontext_base}/retrieval/answer",
                    headers=headers,
                    json=retrieval_payload
                )
                
                if retrieval_response.status_code == 200:
                    return {
                        "source": "qcontext",
                        "insights": retrieval_response.json()
                    }
        except Exception as e:
            print(f"⚠️ Q-Context failed: {str(e)}. Falling back to OpenRouter...")
    
    # Option 2: Fallback to OpenRouter
    if config.openrouter_key:
        try:
            headers = {
                "Authorization": f"Bearer {config.openrouter_key}",
                "Content-Type": "application/json"
            }
            
            prompt = f"""
            Based on this market research data for {startup_name}:
            
            {raw_data}
            
            Provide a structured JSON response with:
            1. strengths: Array of core strengths
            2. weaknesses: Array of key weaknesses/gaps
            3. competitor_advantages: Array of areas where competitors lead
            4. opportunities: Array of market positioning opportunities
            5. positioning_statement: Refined value proposition (2-3 sentences)
            6. differentiation_angle: Unique differentiation (1-2 sentences)
            7. target_audience: Ideal customer profile
            
            Return ONLY valid JSON.
            """
            
            payload = {
                "model": "google/gemini-2.0-flash-exp:free",
                "messages": [{"role": "user", "content": prompt}],
                "response_format": {"type": "json_object"}
            }
            
            response = requests.post(
                f"{config.openrouter_base}/chat/completions",
                headers=headers,
                json=payload
            )
            
            if response.status_code == 200:
                result = response.json()
                content = result["choices"][0]["message"]["content"]
                return {
                    "source": "openrouter_llm",
                    "insights": json.loads(content)
                }
        except Exception as e:
            print(f"❌ OpenRouter failed: {str(e)}")
    
    return {"error": "No API available for insight structuring"}

# Execute insight structuring
insights = insight_structuring_node(competitor_data, sentiment_data, user_input.startup_name)
workflow_state["insights"] = insights

pretty_print("Structured Insights", insights)

## Node 5: Video Script Generation (OpenRouter)

Generate a compelling video script based on insights

In [ ]:
def video_script_generation_node(startup_name: str, insights: Dict) -> Dict[str, Any]:
    """
    Generate video script using OpenRouter
    """
    
    if not config.openrouter_key:
        return {"error": "OpenRouter API key not configured"}
    
    try:
        headers = {
            "Authorization": f"Bearer {config.openrouter_key}",
            "Content-Type": "application/json"
        }
        
        insights_data = insights.get("insights", {})
        
        prompt = f"""
        Create a professional 60-90 second video script for {startup_name} that:
        
        POSITIONING:
        {json.dumps(insights_data, indent=2)}
        
        SCRIPT REQUIREMENTS:
        1. Opening hook (5-10 seconds) - Grab attention with market problem/gap
        2. Strengths showcase (20-30 seconds) - Highlight unique advantages
        3. Address gaps (15-20 seconds) - Turn weaknesses into opportunities
        4. Differentiation (15-20 seconds) - Clear competitive positioning
        5. Call to action (5-10 seconds) - Compelling next step
        
        FORMAT:
        - CEO-level narrative tone
        - Aspirational and trustworthy
        - Include [VISUAL CUE] notes for each section
        - Keep language simple and impactful
        
        Return as JSON with: {{"script": "...", "visual_cues": [...], "duration_estimate": "..."}}
        """
        
        payload = {
            "model": "google/gemini-2.0-flash-exp:free",
            "messages": [{"role": "user", "content": prompt}],
            "response_format": {"type": "json_object"}
        }
        
        response = requests.post(
            f"{config.openrouter_base}/chat/completions",
            headers=headers,
            json=payload
        )
        
        if response.status_code == 200:
            result = response.json()
            content = result["choices"][0]["message"]["content"]
            return json.loads(content)
            
    except Exception as e:
        return {"error": f"Video script generation failed: {str(e)}"}

# Generate video script
video_script = video_script_generation_node(user_input.startup_name, insights)
workflow_state["video_script"] = video_script

pretty_print("Video Script", video_script)

## Node 6: Video Generation (Hera)

Create actual video using Hera API (optional - may take time)

In [ ]:
def video_generation_node(video_script: Dict) -> Dict[str, Any]:
    """
    Generate video using Hera API
    Note: This is optional and may take several minutes
    """
    
    if not config.hera_key:
        return {"status": "skipped", "reason": "Hera API key not configured"}
    
    try:
        headers = {
            "x-api-key": config.hera_key,
            "Content-Type": "application/json"
        }
        
        script_text = video_script.get("script", "")
        
        payload = {
            "prompt": script_text,
            "formats": ["mp4"],
            "aspectRatio": "16:9"
        }
        
        response = requests.post(
            f"{config.hera_base}/videos",
            headers=headers,
            json=payload
        )
        
        if response.status_code in [200, 201]:
            job_result = response.json()
            video_id = job_result.get("id") or job_result.get("video_id")
            
            return {
                "status": "processing",
                "video_id": video_id,
                "message": f"Video generation started. Check status with ID: {video_id}"
            }
        else:
            return {
                "status": "failed",
                "error": response.text
            }
            
    except Exception as e:
        return {
            "status": "error",
            "error": str(e)
        }

# Optional: Generate video (comment out if you want to skip)
# video_result = video_generation_node(video_script)
# workflow_state["video_generation"] = video_result
# pretty_print("Video Generation Status", video_result)

print("⏭️ Video generation skipped (uncomment code above to enable)")

## Node 7: LinkedIn Content Generation (OpenRouter)

Generate 3 LinkedIn posts based on insights

In [ ]:
def linkedin_generation_node(startup_name: str, insights: Dict, competitors: List[str]) -> Dict[str, Any]:
    """
    Generate LinkedIn posts using OpenRouter
    """
    
    if not config.openrouter_key:
        return {"error": "OpenRouter API key not configured"}
    
    try:
        headers = {
            "Authorization": f"Bearer {config.openrouter_key}",
            "Content-Type": "application/json"
        }
        
        insights_data = insights.get("insights", {})
        
        prompt = f"""
        Generate 3 LinkedIn posts for {startup_name} based on these insights:
        
        {json.dumps(insights_data, indent=2)}
        
        COMPETITORS: {', '.join(competitors[:3])}
        
        POST 1 - MARKET INSIGHT:
        - Highlight a gap or trend in the market
        - Position {startup_name} as the solution
        - Use data/insights from competitor analysis
        - 150-200 words
        - Include relevant emoji and hashtags
        
        POST 2 - FOUNDER NARRATIVE:
        - Share the vision/mission
        - Emphasize differentiation and unique approach
        - Personal and authentic tone
        - 150-200 words
        - Include relevant emoji and hashtags
        
        POST 3 - PRODUCT-LED:
        - Feature comparison vs competitors
        - Highlight specific benefits and advantages
        - Include social proof or metrics if applicable
        - 150-200 words
        - Include relevant emoji and hashtags
        
        Return as JSON: {{"posts": [{{"type": "...", "content": "...", "hashtags": [...]}}]}}
        """
        
        payload = {
            "model": "google/gemini-2.0-flash-exp:free",
            "messages": [{"role": "user", "content": prompt}],
            "response_format": {"type": "json_object"}
        }
        
        response = requests.post(
            f"{config.openrouter_base}/chat/completions",
            headers=headers,
            json=payload
        )
        
        if response.status_code == 200:
            result = response.json()
            content = result["choices"][0]["message"]["content"]
            return json.loads(content)
            
    except Exception as e:
        return {"error": f"LinkedIn generation failed: {str(e)}"}

# Generate LinkedIn posts
linkedin_posts = linkedin_generation_node(user_input.startup_name, insights, competitors)
workflow_state["linkedin_posts"] = linkedin_posts

pretty_print("LinkedIn Posts", linkedin_posts)

## Node 8: Final Report Generation

Compile all outputs into a structured report

In [ ]:
def generate_final_report(workflow_state: Dict) -> Dict[str, Any]:
    """
    Compile final report with all deliverables
    """
    
    insights_data = workflow_state["insights"].get("insights", {})
    
    report = {
        "metadata": {
            "startup_name": workflow_state["input"]["startup_name"],
            "generated_at": workflow_state["timestamp"],
            "workflow_version": "1.0"
        },
        
        "company_overview": {
            "name": workflow_state["input"]["startup_name"],
            "industry": workflow_state["input"].get("industry", "N/A"),
            "target_audience": workflow_state["input"].get("target_audience", "N/A")
        },
        
        "competitive_landscape": {
            "source": workflow_state["competitor_analysis"].get("source"),
            "competitors": workflow_state["competitor_analysis"].get("data", {})
        },
        
        "strength_analysis": {
            "core_strengths": insights_data.get("strengths", []),
            "unique_capabilities": insights_data.get("differentiation_angle", "")
        },
        
        "weakness_analysis": {
            "narrative_gaps": insights_data.get("weaknesses", []),
            "areas_for_improvement": insights_data.get("opportunities", [])
        },
        
        "competitor_advantage": {
            "where_competitors_lead": insights_data.get("competitor_advantages", [])
        },
        
        "positioning": {
            "value_proposition": insights_data.get("positioning_statement", ""),
            "differentiation": insights_data.get("differentiation_angle", ""),
            "target_audience": insights_data.get("target_audience", "")
        },
        
        "content_deliverables": {
            "video_script": workflow_state.get("video_script", {}),
            "linkedin_posts": workflow_state.get("linkedin_posts", {})
        }
    }
    
    return report

# Generate final report
final_report = generate_final_report(workflow_state)
workflow_state["final_report"] = final_report

pretty_print("FINAL EDGEELEVATE REPORT", final_report)

## Export Report

Save the final report to a JSON file

In [ ]:
import os
from pathlib import Path

# Create output directory
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

# Generate filename
startup_name_clean = user_input.startup_name.replace(" ", "_").lower()
timestamp = time.strftime("%Y%m%d_%H%M%S")
output_file = output_dir / f"edgeelevate_report_{startup_name_clean}_{timestamp}.json"

# Save report
with open(output_file, "w") as f:
    json.dump(final_report, f, indent=2)

print(f"\n✅ Report saved to: {output_file}")
print(f"\n📊 Workflow complete for {user_input.startup_name}!")

## Summary

View execution summary and next steps

In [ ]:
print("\n" + "="*60)
print("EDGEELEVATE WORKFLOW SUMMARY")
print("="*60)

print(f"\n📋 Startup Analyzed: {user_input.startup_name}")
print(f"⏰ Completed At: {workflow_state['timestamp']}")

print("\n✅ Nodes Executed:")
nodes = [
    ("User Input", "input" in workflow_state),
    ("Competitor Analysis", "competitor_analysis" in workflow_state),
    ("Sentiment Analysis", "sentiment_analysis" in workflow_state),
    ("Insight Structuring", "insights" in workflow_state),
    ("Video Script Generation", "video_script" in workflow_state),
    ("LinkedIn Content", "linkedin_posts" in workflow_state),
    ("Final Report", "final_report" in workflow_state)
]

for node_name, status in nodes:
    status_icon = "✅" if status else "❌"
    print(f"  {status_icon} {node_name}")

print("\n📦 Deliverables Generated:")
if "video_script" in workflow_state:
    print("  ✅ Video Script")
if "linkedin_posts" in workflow_state:
    posts = workflow_state["linkedin_posts"].get("posts", [])
    print(f"  ✅ LinkedIn Posts ({len(posts)} posts)")
if "final_report" in workflow_state:
    print("  ✅ Structured Report")

print("\n🎯 Next Steps:")
print("  1. Review the generated report and insights")
print("  2. Refine LinkedIn posts for your brand voice")
print("  3. Use video script to create actual video (via Hera or other tools)")
print("  4. Implement positioning recommendations")
print("  5. Track competitor changes over time")

print("\n" + "="*60)